# 📊 Extrato Atendimento Central Metrics: DuckDB vs AWS Athena

This notebook connects to both the local **DuckDB** database (`huntington_data_lake.duckdb`) and **AWS Athena (Production)** (`gold_huntington_prod`) to execute data quality and validation queries on the `extrato_atendimento_central` table.

### Key Differences Identified during Research:
1. **Table Names**: Local table is `gold.extrato_atendimento_central` (singular), Cloud table is `gold_huntington_prod.clinisys_extrato_atendimentos_central` (plural).
2. **Hidden Records Filter (`oculto = '0'`)**: Local loader applies a filter of `oculto = '0'` while the cloud loader includes all records. This explains the ~28k record difference (Cloud has more rows).
3. **Data Type Mismatches**: 
   - `prontuario` is double locally and bigint in cloud.
   - `data` / `data_agendamento_original` are timestamp locally and date in cloud.
   - `inicio` is timestamp locally and string in cloud.

This notebook reconciles:
1. **Table Schema Audit & Mapping**
2. **Global Row Count & Min/Max Date Bounds**
3. **Yearly Row Count & Patient Count Breakdown**
4. **Completeness Comparison (Non-null / Non-empty rates)**
5. **Row-level Value Discrepancy Diagnostics (Overlapping subset)**

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
import re
import datetime
import unicodedata
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_SCHEMA = 'gold_huntington_prod'

print("Connecting to DuckDB...")
duck_con = duckdb.connect(DUCKDB_PATH, read_only=True)

print(f"Connecting to AWS Athena ({ATHENA_SCHEMA})...")
try:
    ath_con = connect(
        region_name=ATHENA_REGION,
        work_group=ATHENA_WORKGROUP,
        schema_name=ATHENA_SCHEMA
    )
    ath_cur = ath_con.cursor()
    print("Successfully connected to AWS Athena.")
except Exception as e:
    print(f"Warning: Could not connect to Athena: {e}")
    ath_cur = None

## 🔍 Part 1: Schema Comparison

Checking local and cloud schemas to identify data type differences.

In [ ]:
df_duck_schema = duck_con.execute("DESCRIBE gold.extrato_atendimento_central").df()

if ath_cur is not None:
    ath_cur.execute("DESCRIBE gold_huntington_prod.clinisys_extrato_atendimentos_central")
    cloud_schema = []
    for r in ath_cur.fetchall():
        if r[0] and not r[0].startswith('#') and r[0].strip() != '':
            parts = r[0].split('\t')
            col_name = parts[0].strip()
            col_type = parts[1].strip() if len(parts) > 1 else 'unknown'
            cloud_schema.append((col_name, col_type))
    df_ath_schema = pd.DataFrame(cloud_schema, columns=['column_name', 'column_type'])
    
    schema_comp = pd.merge(
        df_duck_schema[['column_name', 'column_type']],
        df_ath_schema[['column_name', 'column_type']],
        on='column_name',
        suffixes=('_duck', '_ath'),
        how='outer'
    )
    print("--- Schema Comparison (DuckDB vs Athena) ---")
    display(schema_comp)
else:
    print("Athena schema not available.")

## 🔍 Part 2: Load Data for Reconciliations

Querying full tables from both local and cloud databases.

In [ ]:
print("Loading local DuckDB data...")
df_duck = duck_con.execute("SELECT * FROM gold.extrato_atendimento_central").df()
print(f"Loaded {len(df_duck):,} rows from local DuckDB gold.extrato_atendimento_central.")

if ath_con is not None:
    print("Loading cloud AWS Athena data (this may take a moment)...")
    df_ath = pd.read_sql("SELECT * FROM gold_huntington_prod.clinisys_extrato_atendimentos_central", ath_con)
    print(f"Loaded {len(df_ath):,} rows from cloud AWS Athena gold_huntington_prod.clinisys_extrato_atendimentos_central.")
else:
    df_ath = None
    print("AWS Athena connection is not available.")

## 🔍 Part 3: Normalization & Date/Year Extractions

Extracting year from `data` in both datasets to perform yearly breakdown analyses.

In [ ]:
def extract_year(val):
    if pd.isna(val) or val is None:
        return None
    if isinstance(val, (datetime.date, datetime.datetime)):
        return val.year
    try:
        return pd.to_datetime(val).year
    except:
        return None

df_duck['year'] = df_duck['data'].apply(extract_year)
if df_ath is not None:
    df_ath['year'] = df_ath['data'].apply(extract_year)
    print("Year extraction completed successfully.")

## ⚖️ Part 4: Overall Volume & Cardinality Reconciliation

Comparing global metrics including row count, unique patient count, unique doctor count, and date ranges.

In [ ]:
if df_ath is not None:
    duck_overall = {
        "source": "Local (DuckDB)",
        "total_rows": len(df_duck),
        "unique_patients": df_duck["paciente_codigo"].nunique(),
        "unique_medicos": df_duck["medico"].nunique(),
        "unique_agendamentos": df_duck["agendamento_id"].nunique(),
        "min_date": str(df_duck["data"].min())[:10],
        "max_date": str(df_duck["data"].max())[:10]
    }
    ath_overall = {
        "source": "Cloud (Athena)",
        "total_rows": len(df_ath),
        "unique_patients": df_ath["paciente_codigo"].nunique(),
        "unique_medicos": df_ath["medico"].nunique(),
        "unique_agendamentos": df_ath["agendamento_id"].nunique(),
        "min_date": str(df_ath["data"].min())[:10],
        "max_date": str(df_ath["data"].max())[:10]
    }
    df_overall = pd.DataFrame([duck_overall, ath_overall])
    print("--- Overall Table Metrics Comparison ---")
    display(df_overall)
else:
    print("Athena data not available.")

## ⚖️ Part 5: Side-by-Side Yearly Breakdown

Analyzing volume difference per year to look for consistent ingestion drift or anomalies.

In [ ]:
if df_ath is not None:
    duck_yr = df_duck.groupby("year").agg(
        duck_rows=("agendamento_id", "count"),
        duck_patients=("paciente_codigo", "nunique")
    ).reset_index()
    
    ath_yr = df_ath.groupby("year").agg(
        ath_rows=("agendamento_id", "count"),
        ath_patients=("paciente_codigo", "nunique")
    ).reset_index()
    
    df_yr_comp = pd.merge(duck_yr, ath_yr, on="year", how="outer").sort_values("year").fillna(0)
    for col in df_yr_comp.columns:
        if col != "year":
            df_yr_comp[col] = df_yr_comp[col].astype(int)
            
    df_yr_comp['row_diff'] = df_yr_comp['ath_rows'] - df_yr_comp['duck_rows']
    df_yr_comp['row_diff_pct'] = round((df_yr_comp['row_diff'] / df_yr_comp['duck_rows']) * 100, 2)
    
    print("--- Side-by-side Yearly Comparison ---")
    display(df_yr_comp)
else:
    print("Athena data not available.")

## ⚖️ Part 6: Column Data Completeness Side-by-Side

Reconciling completion rates (percentage of non-null / non-empty values) for all shared columns.

In [ ]:
if df_ath is not None:
    common_cols = sorted(list(set(df_duck.columns) & set(df_ath.columns)))
    completion_records = []
    for col in common_cols:
        if col in ["year"]:
            continue
        
        # DuckDB non-null / non-blank
        duck_valid = df_duck[col].dropna()
        duck_valid = duck_valid[duck_valid.astype(str).str.strip() != ""] if df_duck[col].dtype == object else duck_valid
        duck_pct = (len(duck_valid) / len(df_duck)) * 100.0 if len(df_duck) > 0 else 0.0
        
        # Athena non-null / non-blank
        ath_valid = df_ath[col].dropna()
        ath_valid = ath_valid[ath_valid.astype(str).str.strip() != ""] if df_ath[col].dtype == object else ath_valid
        ath_pct = (len(ath_valid) / len(df_ath)) * 100.0 if len(df_ath) > 0 else 0.0
        
        completion_records.append({
            "column": col,
            "duck_non_null": len(duck_valid),
            "duck_completion_pct": round(duck_pct, 2),
            "ath_non_null": len(ath_valid),
            "ath_completion_pct": round(ath_pct, 2),
            "diff_pct": round(abs(duck_pct - ath_pct), 2)
        })
        
    df_completion = pd.DataFrame(completion_records).sort_values("diff_pct", ascending=False)
    print("--- Column Data Completion Summary (%) ---")
    display(df_completion)
else:
    print("Athena data not available.")

## ⚖️ Part 7: Row-Level Value Discrepancy Analysis (Overlapping Subset)

Running inner join on `agendamento_id` to evaluate whether data is mathematically equivalent for identical appointments.

In [ ]:
def normalize_val(value, column_name=None):
    if pd.isna(value) or value is None:
        return None
    
    # Ints / IDs: convert to integer
    if column_name in ["prontuario", "paciente_codigo", "medico", "medico2", "evento", "centro_custos", "agenda", "confirmado", "agendamento_id"]:
        try:
            return int(float(value))
        except (ValueError, TypeError):
            return value
            
    # Dates: format to string 'YYYY-MM-DD'
    if column_name in ["data", "data_agendamento_original"]:
        try:
            if isinstance(value, str):
                return value[:10]
            return value.strftime("%Y-%m-%d")
        except:
            return value
            
    # Time: compare 'HH:MM'
    if column_name == "inicio":
        try:
            if isinstance(value, str):
                t_str = value.strip()
                if ' ' in t_str:
                    t_str = t_str.split(' ')[1]
                return t_str[:5]
            return value.strftime("%H:%M")
        except:
            return value

    # Strings: lowercase, remove accents, strip whitespace
    if isinstance(value, str):
        val = value.lower().strip()
        val = "".join(c for c in unicodedata.normalize('NFD', val) if unicodedata.category(c) != 'Mn')
        val = re.sub(r'\s+', ' ', val)
        return val
        
    return value

if df_ath is not None:
    print("--- Row-level inner join stats ---")
    merged = pd.merge(
        df_duck,
        df_ath,
        on=["agendamento_id"],
        suffixes=("_duck", "_ath"),
        how="inner"
    )
    print(f"Inner joined matching rows count: {len(merged):,}")
    print(f"Rows only in DuckDB (Local): {len(df_duck) - len(merged):,}")
    print(f"Rows only in Athena (Cloud): {len(df_ath) - len(merged):,}")
    
    value_discrepancy_cols = [c for c in common_cols if c not in ["agendamento_id", "year"]]
    discrepancy_counts = {}
    
    for col in value_discrepancy_cols:
        col_duck = f"{col}_duck"
        col_ath = f"{col}_ath"
        
        norm_duck = merged[col_duck].apply(lambda x: normalize_val(x, col))
        norm_ath = merged[col_ath].apply(lambda x: normalize_val(x, col))
        
        mask = (norm_duck != norm_ath) & ~(norm_duck.isna() & norm_ath.isna())
        diff_count = mask.sum()
        
        if diff_count > 0:
            discrepancy_counts[col] = diff_count
            
    if discrepancy_counts:
        print("\nFound value discrepancies in the following columns:")
        for col, count in sorted(discrepancy_counts.items(), key=lambda x: x[1], reverse=True):
            print(f" - {col}: {count:,} rows with differences ({round(count * 100.0 / len(merged), 2)}% of matched)")
            
            col_duck = f"{col}_duck"
            col_ath = f"{col}_ath"
            norm_duck = merged[col_duck].apply(lambda x: normalize_val(x, col))
            norm_ath = merged[col_ath].apply(lambda x: normalize_val(x, col))
            mask = (norm_duck != norm_ath) & ~(norm_duck.isna() & norm_ath.isna())
            samples = merged[mask][["agendamento_id", col_duck, col_ath]].head(5)
            print("   Sample discrepancies:")
            display(samples)
            print()
    else:
        print("\nSuccess! No value discrepancies found among the inner joined rows.")
else:
    print("Athena data not available.")